# 05 — Document Revision Workflow Demo (the manual "supersede" workaround)

This notebook is the runnable companion to chapter 05
(`05-document-lifecycle-versioning-and-revisions.md`) — the chapter that directly answers the real
interview question this whole course rebuild was triggered by: *"for revised versions of the same
document, how are you handling those?"*

It does two things:

1. **Demonstrates, against a mocked `INGEST_API` client, that today's system has no versioning** —
   uploading the "same" document twice produces two independent documents with two different IDs, with
   nothing linking them.
2. **Demonstrates the manual three-step "supersede" workaround** that chapter 05 describes as
   achievable *today*, with existing endpoints: search by title + business line, remove the stale
   match, then upload the revision.

A short final section sketches the **proposed** `document_group_id`/`supersedes_document_id` design
from chapter 05 as a dataclass and a couple of query helpers — clearly labeled as a proposal, not
something implemented in the real service.

Everything in this notebook is a self-contained, offline mock — no real `INGEST_API`, no network calls,
no real Azure credentials, matching this course's "runs offline" rule for notebooks. The mock
deliberately mirrors the **real** endpoint shapes documented in `FULL_ARCHITECTURE.md` section 4 and
`app.py`'s `upload_files()`/`search_ingested_documents()`/`remove_ingested_document()`:
`batch-initialize` returns a fresh `doc_id`, `ingest` returns a fresh `documentID`, `search` supports
filtering by `document_title`, and `remove` deletes by `parent_document_id`.


## 1. A mocked `INGEST_API` client, matching the real endpoint shapes

`MockIngestAPI` stands in for the real external ingestion API this service calls
(`INGEST_API` in `app.py`). It intentionally has **no title-based dedup, no supersede logic, and no
version concept anywhere** — matching the real system exactly (chapter 05, Part 1): every `ingest()`
call is handed a brand-new UUID, regardless of whether a document with the same title already exists.


In [1]:
import uuid
from dataclasses import dataclass, field
from typing import Dict, List, Optional


@dataclass
class IngestedDocument:
    document_id: str
    title: str
    business_line: str
    classification: str
    content: str  # stand-in for file bytes
    status: str = "SUCCESSFUL"


class MockIngestAPI:
    """
    Offline stand-in for the real external INGEST_API this service calls.
    Mirrors the real shape: batch-initialize + ingest both hand out brand-new
    IDs every time, search filters by title/business_line, remove deletes by
    parent_document_id. No version concept anywhere -- exactly like the real
    system (chapter 05, Part 1).
    """

    def __init__(self):
        self._documents: Dict[str, IngestedDocument] = {}

    def batch_initialize(self, business_line: str, titles: List[str]) -> List[str]:
        """Mirrors POST ingest/batch-initialize/{use_case}/HEXA -- always returns fresh doc_ids."""
        return [str(uuid.uuid4()) for _ in titles]

    def ingest(self, doc_id: str, title: str, business_line: str, classification: str, content: str) -> str:
        """Mirrors POST ingest/{use_case}/HEXA -- always creates a brand-new document, no dedup check."""
        document_id = str(uuid.uuid4())  # the real service returns a fresh documentID on every call
        self._documents[document_id] = IngestedDocument(
            document_id=document_id,
            title=title,
            business_line=business_line,
            classification=classification,
            content=content,
        )
        return document_id

    def search(self, document_title: Optional[str] = None, business_line: Optional[str] = None) -> List[IngestedDocument]:
        """Mirrors GET ingest/HEXA (via /search-ingested-documents/{use_case}), filtered client-side."""
        results = list(self._documents.values())
        if document_title is not None:
            results = [d for d in results if d.title == document_title]
        if business_line is not None:
            results = [d for d in results if d.business_line == business_line]
        return results

    def remove(self, parent_document_id: str) -> bool:
        """Mirrors DELETE ingest/{use_case}/HEXA?parent_document_id=...&strict=true."""
        return self._documents.pop(parent_document_id, None) is not None


ingest_api = MockIngestAPI()
print("MockIngestAPI ready -- no version concept, exactly like the real INGEST_API this service calls.")


MockIngestAPI ready -- no version concept, exactly like the real INGEST_API this service calls.


## 2. Proving there's no versioning today: upload the "same" document twice

This reproduces exactly what happens today if a user uploads a revised version of a document without
doing anything special — precisely the scenario the real interview question was about.


In [2]:
def upload_document(api: MockIngestAPI, title: str, business_line: str, classification: str, content: str) -> str:
    """
    Mirrors app.py's upload_files(): batch-initialize, then ingest -- the same two-step
    call sequence the real service makes for every non-IWPB department (and, for IWPB,
    once an approver has approved the document).
    """
    [doc_id] = api.batch_initialize(business_line, [title])
    document_id = api.ingest(doc_id, title, business_line, classification, content)
    return document_id


# Upload v1 of a policy document
v1_id = upload_document(
    ingest_api,
    title="AML_Policy_2026",
    business_line="GENERAL",
    classification="INTERNAL",
    content="v1 content -- original policy text",
)
print(f"Uploaded v1: documentID={v1_id}")

# Someone later uploads a REVISED version, with the same title, the naive way --
# exactly what "just upload the new file" looks like today, with no extra steps.
v2_id_naive = upload_document(
    ingest_api,
    title="AML_Policy_2026",
    business_line="GENERAL",
    classification="INTERNAL",
    content="v2 content -- updated policy text",
)
print(f"Uploaded v2 (naive, no cleanup): documentID={v2_id_naive}")

print()
print("Search results for this title now:")
for doc in ingest_api.search(document_title="AML_Policy_2026"):
    print(f"  - documentID={doc.document_id}  content={doc.content!r}")

assert v1_id != v2_id_naive, "Two independent document IDs, exactly as the real system behaves."
assert len(ingest_api.search(document_title="AML_Policy_2026")) == 2, (
    "Both the stale v1 and the new v2 are independently discoverable via search -- "
    "there is no linkage, no supersede, and no automatic cleanup."
)
print()
print("Confirmed: uploading a revision the naive way leaves BOTH documents discoverable,")
print("with no relationship between them -- this is exactly today's real behavior (chapter 05, Part 1).")


Uploaded v1: documentID=5078a1e8-a7e4-41b9-bcaf-c8aaedc58543
Uploaded v2 (naive, no cleanup): documentID=a70e5c9f-1beb-459a-b24d-24a1d1a21677

Search results for this title now:
  - documentID=5078a1e8-a7e4-41b9-bcaf-c8aaedc58543  content='v1 content -- original policy text'
  - documentID=a70e5c9f-1beb-459a-b24d-24a1d1a21677  content='v2 content -- updated policy text'

Confirmed: uploading a revision the naive way leaves BOTH documents discoverable,
with no relationship between them -- this is exactly today's real behavior (chapter 05, Part 1).


## 3. The manual "supersede" workaround (chapter 05, Part 2) — runnable end to end

Three steps, using only operations the real service's existing endpoints already support:
**search** by title + business line to find the stale version, **remove** it, then **upload** the
revision. This is entirely client-driven — nothing in the real service enforces or automates it — but
it works today, with zero changes to the real system.


In [3]:
def supersede_document(
    api: MockIngestAPI,
    title: str,
    business_line: str,
    classification: str,
    new_content: str,
) -> str:
    """
    The manual workaround from chapter 05, Part 2, implemented against the mock API:
      1. Search by title + business_line to find the stale version(s).
      2. Remove each stale match.
      3. Upload the revision as a normal new document.

    Mirrors calling, in order: GET /search-ingested-documents/{use_case}?document_title=...,
    POST /remove-ingested-document/{use_case}, then POST /upload-files/{use_case} --
    exactly the three real endpoints chapter 05 names as already existing and already working.
    """

    # Step 1: find the stale version(s)
    stale_matches = api.search(document_title=title, business_line=business_line)

    # Step 2: remove each one
    removed_ids = []
    for stale in stale_matches:
        removed = api.remove(stale.document_id)
        if removed:
            removed_ids.append(stale.document_id)

    # Step 3: upload the revision as a normal new document
    new_document_id = upload_document(api, title, business_line, classification, new_content)

    return new_document_id, removed_ids


# Start from a clean slate to demonstrate the workaround in isolation
clean_api = MockIngestAPI()

v1_id = upload_document(
    clean_api, title="KYC_Checklist", business_line="GENERAL",
    classification="INTERNAL", content="v1 -- original checklist",
)
print(f"v1 uploaded: documentID={v1_id}")

v2_id, removed = supersede_document(
    clean_api, title="KYC_Checklist", business_line="GENERAL",
    classification="INTERNAL", new_content="v2 -- revised checklist with new KYC fields",
)
print(f"Supersede workaround ran: removed stale ID(s) {removed}, new documentID={v2_id}")

remaining = clean_api.search(document_title="KYC_Checklist")
print()
print("Search results for this title now:")
for doc in remaining:
    print(f"  - documentID={doc.document_id}  content={doc.content!r}")

assert len(remaining) == 1, "Exactly one current version should remain after the workaround."
assert remaining[0].document_id == v2_id, "The remaining document should be the new revision."
assert v1_id in removed, "The stale v1 document should have been removed."
print()
print("Confirmed: the manual supersede workaround leaves exactly one current version discoverable.")
print("Caveat (chapter 05): the original documentID is gone -- anything that referenced v1's ID")
print("(a link, a citation, an audit log entry) now points at a removed document.")


v1 uploaded: documentID=e7307bd5-7d91-44fe-bddb-072e5b00634c
Supersede workaround ran: removed stale ID(s) ['e7307bd5-7d91-44fe-bddb-072e5b00634c'], new documentID=d1a1d7ad-9e10-4207-9e40-2345bbc2035e

Search results for this title now:
  - documentID=d1a1d7ad-9e10-4207-9e40-2345bbc2035e  content='v2 -- revised checklist with new KYC fields'

Confirmed: the manual supersede workaround leaves exactly one current version discoverable.
Caveat (chapter 05): the original documentID is gone -- anything that referenced v1's ID
(a link, a citation, an audit log entry) now points at a removed document.


## 4. The proposed real design — a sketch, not implemented (chapter 05, Part 4)

This section is intentionally **not** exercised against the mock `INGEST_API` above — it's a stub
showing the shape of a genuine fix, for contrast with the workaround above. Two fields would need to
exist on whatever tracks "a document" (today, that's only real for IWPB's `IWPBDocumentWorkflow`; a
generalized version would need an equivalent table for every department, which doesn't exist today
either):


In [4]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional


@dataclass
class ProposedDocumentRevision:
    """
    NOT IMPLEMENTED -- a proposed extension to this service's document model (chapter 05, Part 4),
    sketched here purely for contrast with the manual workaround demonstrated above.

    document_group_id       -- stable across every revision of "the same logical document."
    supersedes_document_id  -- the specific prior revision this upload replaces, if any.
    """

    document_id: str
    document_group_id: str
    title: str
    business_line: str
    content: str
    supersedes_document_id: Optional[str] = None
    created_at: datetime = field(default_factory=datetime.utcnow)


def get_latest_version(revisions: list["ProposedDocumentRevision"], document_group_id: str):
    """
    Proposed query: 'the latest version' becomes well-defined once document_group_id exists --
    the row in this group that no other row in the same group supersedes.
    """
    group = [r for r in revisions if r.document_group_id == document_group_id]
    superseded_ids = {r.supersedes_document_id for r in group if r.supersedes_document_id}
    current = [r for r in group if r.document_id not in superseded_ids]
    return max(current, key=lambda r: r.created_at) if current else None


def get_revision_history(revisions: list["ProposedDocumentRevision"], document_group_id: str):
    """Proposed query: full version history, oldest first."""
    group = [r for r in revisions if r.document_group_id == document_group_id]
    return sorted(group, key=lambda r: r.created_at)


# A small illustrative walk-through of the proposed model (not the mock INGEST_API -- this is a
# separate, local sketch, since INGEST_API itself would need to gain a version concept too, per
# chapter 05's honest caveat that this service can't fully solve versioning unilaterally).
group_id = str(uuid.uuid4())

rev1 = ProposedDocumentRevision(
    document_id=str(uuid.uuid4()), document_group_id=group_id,
    title="AML_Policy_2026", business_line="GENERAL", content="v1 content",
)
rev2 = ProposedDocumentRevision(
    document_id=str(uuid.uuid4()), document_group_id=group_id,
    title="AML_Policy_2026", business_line="GENERAL", content="v2 content",
    supersedes_document_id=rev1.document_id,
)

all_revisions = [rev1, rev2]

latest = get_latest_version(all_revisions, group_id)
history = get_revision_history(all_revisions, group_id)

print(f"Latest version content: {latest.content!r}")
print(f"Revision history (oldest first): {[r.content for r in history]}")

assert latest.document_id == rev2.document_id
assert [r.document_id for r in history] == [rev1.document_id, rev2.document_id]
print()
print("This is the proposed design from chapter 05 -- 'get latest' and 'show history' become")
print("well-defined queries. It is NOT implemented in the real service, and INGEST_API/HEXA")
print("would need an equivalent concept too for a complete, non-bypassable fix.")


Latest version content: 'v2 content'
Revision history (oldest first): ['v1 content', 'v2 content']

This is the proposed design from chapter 05 -- 'get latest' and 'show history' become
well-defined queries. It is NOT implemented in the real service, and INGEST_API/HEXA
would need an equivalent concept too for a complete, non-bypassable fix.


<string>:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).


## Summary

| Section | Demonstrates | Matches |
|---|---|---|
| 1 | A mocked `INGEST_API` with no dedup/version logic | The real service's actual behavior (chapter 05, Part 1) |
| 2 | Uploading a "revision" naively creates two unlinked documents | Confirmed by reading `upload_files()` directly |
| 3 | The manual search-remove-upload workaround, running end to end | Chapter 05, Part 2 — achievable today with existing endpoints |
| 4 | A proposed `document_group_id`/`supersedes_document_id` design | Chapter 05, Part 4 — explicitly a proposal, not implemented |

The gap between sections 3 and 4 *is* the answer to "for revised versions of the same document, how
are you handling those" — section 3 is what's genuinely possible today, section 4 is what a real fix
would look like, and neither should be presented as more than it is.
